# 🛡️ Raqeeb – Weapon Detection with YOLO
Training and evaluation of the weapon detection model (pistol & knife) for the Raqeeb graduation project.
- **Dataset:** Weapon Detection Dataset – Kaggle (`mehmetcubukcu/weapon-detection`), CC BY-SA 4.0
- **Model:** YOLO11n
- **Saved model & results:** `MyDrive/Raqeeb/yolo/`



### Cell 1 – Download the dataset
Downloads the weapon detection dataset (~1.7 GB) from Kaggle using `kagglehub`. The variable `path` stores its location.
⚠️ The data is deleted when the Colab session ends, so it must be re-downloaded every session(~2 min).

In [ ]:
!pip install -q kagglehub
import kagglehub

path = kagglehub.dataset_download("mehmetcubukcu/weapon-detection")
print(path)

100%|██████████| 1.71G/1.71G [00:21<00:00, 85.2MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/mehmetcubukcu/weapon-detection/versions/1


### Cell 2 – Locate files and prepare `data.yaml`
Finds `data.yaml` inside the extra `weapon-detection` folder and stores its folder in `root`.
Each split (train / val / test) contains `images` and `labels`.
Writes a corrected `/content/data.yaml` with the right paths for YOLO.

In [ ]:
import os, glob, yaml

yaml_files = glob.glob(f"{path}/**/data.yaml", recursive=True)
print(yaml_files)
root = os.path.dirname(yaml_files[0])

for split in ["train", "val", "test"]:
    print(split, os.listdir(f"{root}/{split}"))

cfg = yaml.safe_load(open(f"{root}/data.yaml"))
print(cfg)

cfg["path"] = root
cfg["train"] = "train/images"
cfg["val"] = "val/images"
cfg["test"] = "test/images"
yaml.safe_dump(cfg, open("/content/data.yaml", "w"))
print("تم ✅")

['/root/.cache/kagglehub/datasets/mehmetcubukcu/weapon-detection/versions/1/weapon-detection/data.yaml']
train ['images', 'labels']
val ['images', 'labels']
test ['images', 'labels']
{'test': '../test/images', 'train': '../train/images', 'val': '../val/images', 'names': ['pistol', 'smartphone', 'knife', 'monedero', 'billete', 'tarjeta'], 'nc': 6}
تم ✅


### Cell 3 – Rename classes to English
Original class names were in Spanish. Renamed to: `pistol, smartphone, knife, purse, bill, card`.
Class IDs are fixed: 0 = pistol, 2 = knife.

In [ ]:
cfg["names"] = ["pistol", "smartphone", "knife", "purse", "bill", "card"]
yaml.safe_dump(cfg, open("/content/data.yaml", "w"))
print(open("/content/data.yaml").read())

names:
- pistol
- smartphone
- knife
- purse
- bill
- card
nc: 6
path: /root/.cache/kagglehub/datasets/mehmetcubukcu/weapon-detection/versions/1/weapon-detection
test: test/images
train: train/images
val: val/images



### Cell 4 – Mount Google Drive
Mounts Drive so the model and results are saved permanently in `MyDrive/Raqeeb/yolo/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

### Cell 5 – Train YOLO11n
- Model: `yolo11n` (smallest/fastest YOLO, suitable for Raspberry Pi 5)
- 50 epochs, image size 640, batch 16 — took ~1h 20min on a T4 GPU
- Best weights saved to: `MyDrive/Raqeeb/yolo/weapon_v1/weights/best.pt`
- Validation result: **mAP50 = 0.88** (pistol 0.899, knife 0.979)


In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data="/content/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/Raqeeb/yolo",
    name="weapon_v1",
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, de

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79fc49e19080>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

### Cell 6 – First test evaluation (first session)
Loads `best.pt` and evaluates it on the test set. Results saved in `weapon_v1_test`.
The final results used in the report are in Cell 9.

In [ ]:
from ultralytics import YOLO

best = YOLO("/content/drive/MyDrive/Raqeeb/yolo/weapon_v1/weights/best.pt")
test_metrics = best.val(
    data="/content/data.yaml",
    split="test",
    project="/content/drive/MyDrive/Raqeeb/yolo",
    name="weapon_v1_test",
)

---
## ▶️ START HERE every time you reopen the notebook
### Cell 7 – Session setup (replaces Cells 1–4)
Installs libraries, mounts Drive, downloads the data, creates `data.yaml`, and loads the trained model into `best`.
No training needed — takes ~2 minutes.

In [ ]:
!pip install -q kagglehub ultralytics
import kagglehub, os, glob, yaml
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')

path = kagglehub.dataset_download("mehmetcubukcu/weapon-detection")
root = os.path.dirname(glob.glob(f"{path}/**/data.yaml", recursive=True)[0])
cfg = {
    "path": root,
    "train": "train/images", "val": "val/images", "test": "test/images",
    "names": ["pistol", "smartphone", "knife", "purse", "bill", "card"], "nc": 6,
}
yaml.safe_dump(cfg, open("/content/data.yaml", "w"))

best = YOLO("/content/drive/MyDrive/Raqeeb/yolo/weapon_v1/weights/best.pt")
print("جاهز ✅")

### Cell 8 – Class distribution
| Class | train | val | test |
|---|---|---|---|
| knife | 1,825 | 241 | 211 |
| pistol | 1,425 | 50 | 35 |
| smartphone | 575 | 58 | 82 |
| purse | 530 | 38 | 33 |
| bill | 425 | 28 | 24 |
| card | 222 | 35 | 22 |
| **Total** | **5,002** | **450** | **407** |

Notes: every image contains exactly one object. Card is the smallest class (explains its weaker results). The test set has only 35 pistols, so pistol metrics may vary.

In [ ]:
from collections import Counter
names = cfg["names"]
for split in ["train", "val", "test"]:
    c = Counter()
    for f in glob.glob(f"{root}/{split}/labels/*.txt"):
        for line in open(f):
            if line.strip(): c[int(line.split()[0])] += 1
    print(split, {names[k]: v for k, v in sorted(c.items())})

### Cell 9 – Final test evaluation ✅ (official results for the report)
Results and plots saved in `MyDrive/Raqeeb/yolo/weapon_v1_test-2/`

| Class | Precision | Recall | mAP50 |
|---|---|---|---|
| **pistol** | 0.853 | **0.971** | **0.983** |
| **knife** | 0.978 | 0.861 | 0.967 |
| All classes | 0.817 | 0.763 | 0.846 |

34 of 35 pistols detected. Inference speed: 5.5 ms/image on GPU.

In [ ]:
test_metrics = best.val(
    data="/content/data.yaml",
    split="test",
    project="/content/drive/MyDrive/Raqeeb/yolo",
    name="weapon_v1_test",
)

### Cell 10 – Confusion matrix
Columns = true class, rows = predicted class.
- No weapon was completely missed (background row is empty for pistol and knife).
- Main confusion: **pistol ↔ smartphone** (9% of pistols predicted as smartphone, 13% of smartphones predicted as pistol).
- Knife is very reliable: 94% correctly classified.

In [ ]:
from IPython.display import Image
Image("/content/drive/MyDrive/Raqeeb/yolo/weapon_v1_test-2/confusion_matrix_normalized.png", width=700)

### Cell 11 – Choosing the alert confidence threshold
Tests different thresholds on the test set using Raqeeb's alert rule: *alert if any pistol or knife is detected above the threshold.*

| Threshold | False alarms | Missed weapons | Weapon recall |
|---|---|---|---|
| 0.25 | 8 | 8 | 97% |
| 0.30 | 6 | 9 | 96% |
| **0.40** | **2** | **12** | **95%** |
| 0.50 | 2 | 21 | 91% |
| 0.70 | 2 | 51 | 79% |

**Decision: threshold = 0.40** → weapon precision 0.99, recall 0.95. False alarms reduced by 75% while missing only 4 more weapons.

In [ ]:
import os, glob

imgs = sorted(glob.glob(f"{root}/test/images/*"))
rows = []
for r in best.predict(imgs, conf=0.05, verbose=False, stream=True):
    lbl = os.path.splitext(r.path.replace("/images/", "/labels/"))[0] + ".txt"
    gt = {int(l.split()[0]) for l in open(lbl) if l.strip()}
    mx = {0: 0.0, 2: 0.0}  # 0 = pistol, 2 = knife
    for c, f in zip(r.boxes.cls.int().tolist(), r.boxes.conf.tolist()):
        if c in mx: mx[c] = max(mx[c], f)
    rows.append((gt, mx))

def score(t, k):
    tp = fp = fn = 0
    for gt, mx in rows:
        if k == "weapon":
            g, a = bool(gt & {0, 2}), max(mx.values()) >= t
        else:
            g, a = k in gt, mx[k] >= t
        tp += g and a; fp += (not g) and a; fn += g and (not a)
    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    return p, r, fp, fn

print(" conf | pistol P   R  | knife P   R  | weapon P   R  | false alarms | missed weapons")
for t in [0.25, 0.3, 0.4, 0.5, 0.6, 0.7]:
    pp, pr, _, _ = score(t, 0); kp, kr, _, _ = score(t, 2); wp, wr, wfp, wfn = score(t, "weapon")
    print(f" {t:.2f} |   {pp:.2f}  {pr:.2f}  |  {kp:.2f}  {kr:.2f} |   {wp:.2f}  {wr:.2f}  |     {wfp:3d}      |      {wfn:3d}")

### Cell 12 – Real-world test (video/images)
Tests the model on our own videos/images recorded with a phone at chest level, to simulate what the Raqeeb body camera will see (distance, lighting, motion).
- Uses the chosen alert threshold: `conf=0.4`
- `save=True` saves a copy of the video/images with the detected boxes drawn on them
- Output saved in: `MyDrive/Raqeeb/yolo/real_test/`

**Test scenarios:** a person holding a kitchen knife · a person holding a smartphone · a normal scene with no objects.

**Results:** _(to be added after the test)_



In [ ]:
results = best.predict("/content/knive.mp4", conf=0.4, save=True,
                       project="/content/yolo", name="real_test")

NameError: name 'best' is not defined

## Cell 13- *Capture* photo & access camera

In [ ]:

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io

def take_photo():
    js = Javascript("""
        async function takePhoto() {
            const video = document.createElement('video');
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(video);
            video.style.display = 'block';
            video.srcObject = stream;
            await video.play();
            await new Promise(resolve => setTimeout(resolve, 2000));
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks().forEach(track => track.stop());
            video.remove();
            return canvas.toDataURL('image/jpeg');
        }
    """)
    display(js)
    data = eval_js("takePhoto()")
    return Image.open(io.BytesIO(b64decode(data.split(',')[1])))

image = take_photo()
display(image)


In [ ]:

results_live = best.predict(
    source=image,
    conf=0.4,
    save=False,
    verbose=False
)

display(results_live[0].plot())


# Cell 14- live capture

In [ ]:

from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from google.colab import output
from base64 import b64decode
import numpy as np
import cv2
import time

# Open laptop camera in the browser
display(Javascript("""
(async () => {
  const old = document.getElementById('raqeeb-live');
  if (old) old.remove();

  const box = document.createElement('div');
  box.id = 'raqeeb-live';

  const video = document.createElement('video');
  video.autoplay = true;
  video.playsInline = true;
  video.style.width = '480px';

  const stop = document.createElement('button');
  stop.textContent = 'Stop Raqeeb';
  stop.style.display = 'block';

  box.appendChild(video);
  box.appendChild(stop);
  document.body.appendChild(box);

  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  video.srcObject = stream;
  await video.play();

  window.raqeebVideo = video;
  window.raqeebStream = stream;
  window.raqeebStop = false;

  stop.onclick = () => {
    window.raqeebStop = true;
    stream.getTracks().forEach(t => t.stop());
    box.remove();
  };
})();
"""))

# Wait until camera is ready
while True:
    try:
        ready = eval_js(
            "!!window.raqeebVideo && window.raqeebVideo.videoWidth > 0"
        )
        if ready:
            break
    except Exception:
        pass
    time.sleep(1)

print("Raqeeb live detection started. Press Stop Raqeeb to finish.")

# Run detection continuously
while True:
    if eval_js("window.raqeebStop"):
        print("Live detection stopped.")
        break

    frame_data = eval_js("""
      (() => {
        const v = window.raqeebVideo;
        const c = document.createElement('canvas');
        c.width = v.videoWidth;
        c.height = v.videoHeight;
        c.getContext('2d').drawImage(v, 0, 0);
        return c.toDataURL('image/jpeg', 0.8);
      })()
    """)

    frame = cv2.imdecode(
        np.frombuffer(b64decode(frame_data.split(',')[1]), np.uint8),
        cv2.IMREAD_COLOR
    )

    result = best.predict(frame, conf=0.4, verbose=False)[0]
    annotated = result.plot()

    output.clear()
    cv2_imshow(annotated)
    time.sleep(0.3)
